# AXI head-to-head: the kept chat WITH COMPACTION against AXI, on the same job
**What this is.** The long-session test (test 22) put AXI against a chat that is never shortened. Agent tools shorten the chat: when it gets long the model is asked to summarise it, and the summary replaces it. This notebook runs that rival on the SAME job (the dense service graph, seed 20261111), the SAME heavy turns and the SAME model (**Gemini 3.8 Flash**) as test 22, in three versions: `S` (generic summary, at 32,000 tokens), `L` (generic summary, at 128,000 tokens) and `V` (the summary is asked to write out the complete current service graph, at 32,000 tokens). 600 steps each, every reply applied as it comes. Arm A of test 22 (AXI) is **not run again**: its first 600 saved steps are read from the repo (the notebook checks the file's hash).

The predictions and the 19 reading rules were written before any call in `docs/COMPACTION_TEST.md` (hash-locked). The test can come out against AXI, and the page says so.

**This test costs real money, and this notebook never turns billing on.** You do that in Google AI Studio. Then: (1) read the price of Gemini 3.8 Flash on the AI Studio price page (dollars per million input tokens, per million output tokens, and per million *cached* input tokens if the page lists one) and type them in Cell 5; (2) run the pilot in Cell 6 (45 turns of arms S and V, a small cost), which prints what the whole test would cost **at your prices**; (3) only then type a budget in dollars (`BUDGET_USD`) and run the rest. The notebook adds up what the saved answers cost and **refuses the next call once the total reaches the budget**; a second guard stops each run at a cap on tokens sent. Everything finished is saved and the same cell carries on.

Run each cell **one at a time, top to bottom**. **Keep this tab open** while a run is going. If it stops for any reason, open the notebook again, run Cells 2 to 5, then run the cell you were on again.

### Cell 1: put your key into Colab Secrets (once; no code to run)
Colab left bar -> key icon (*Secrets*) -> **Add new secret**. Name: `GOOGLE_API_KEY` (exactly). Value: your key from `aistudio.google.com` -> *API Keys*. Switch **Notebook access** on. Never paste the key into a cell or into the chat.

### Cell 2: get the code from GitHub (finish the update commands in Codespaces and `git push` first)

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/gemini_compaction.py"), "the code is not on GitHub yet: finish the update commands (git push) first"
assert os.path.exists("/content/axi/docs/results/axi_gemini_long_session_seed20261111_default.json"), "the saved arm A of test 22 is missing from the copy"
print("code ok")

### Cell 3: install the Gemini library (about 20 seconds)

In [ ]:
!pip -q install -U google-genai

### Cell 4: connect Google Drive (a pop-up asks permission: Connect / Allow). Your progress and every reply are kept there

In [ ]:
import sys
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_gemini_compaction")

### Cell 5: settings. Fill the lines at the top. PAID is True only after you turned billing on. The budget stays empty until you have read the pilot

In [ ]:
import hashlib, json, time
from google.colab import userdata, files
from axi.experiments import gemini_long as G
from axi.experiments import gemini_compaction as C
from axi.experiments.api_llm import QuotaStop, ServiceBusy, BudgetStop

PAID = None          # True ONLY after you turned billing on for this key in Google AI Studio. False = a free key (about 20 calls a day: not enough for the pilot)
PRICE_IN = None      # dollars per 1,000,000 INPUT tokens of Gemini 3.8 Flash, from the AI Studio price page
PRICE_OUT = None     # dollars per 1,000,000 OUTPUT tokens (thinking tokens are billed as output), same page
PRICE_CACHED = None  # dollars per 1,000,000 CACHED input tokens, from the same page. Needed for the two cost lines of the page (18 and 19)
BUDGET_USD = None    # the most you allow this test to cost. Leave None until you have read the pilot in Cell 6. The notebook stops by itself at this amount.
THINKING = None      # None = the model's own default (the same as test 22)

assert PAID in (True, False), "Set PAID: True if you turned billing on, False for a free key."
try: KEY = userdata.get("GOOGLE_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: a Secret named GOOGLE_API_KEY with Notebook access on.")
if PAID: assert PRICE_IN and PRICE_OUT, "Type PRICE_IN and PRICE_OUT (dollars per million tokens) from the AI Studio price page. The notebook has no built-in price on purpose."
TAG = str(THINKING or "default"); CACHE = "axi_gemini_compaction_cache_%s.jsonl" % TAG
print("reply file:", store.restore(CACHE))
guard = dict(budget_usd=BUDGET_USD, price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED) if (PAID and BUDGET_USD) else dict(price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED)
llm, MODEL = G.build_chat_llm(KEY, store.path(CACHE), daily_limit=None if PAID else 20, min_interval=0.3 if PAID else 4.5, on_write=lambda: store.persist(CACHE), thinking=THINKING, **guard)
META = dict(model=MODEL, thinking=THINKING)
sp = llm.spent_usd()
print("paid:", PAID, "| model:", MODEL, "| thinking:", THINKING or "model default", "| budget:", ("$%.2f" % BUDGET_USD) if BUDGET_USD else "NOT SET (pilot only)", "| spent so far with these settings:", ("$%.2f" % sp) if sp is not None else "-")
RESET = "The free quota resets at midnight Pacific time. Then open this notebook again, run Cells 2 to 5, and run this cell again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once. Do not run it again and again."
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e))
        if isinstance(e, BudgetStop): pass
        elif isinstance(e, ServiceBusy): print(BUSY)
        else: print(RESET)
        return None
def need_budget(): assert PAID and BUDGET_USD, "Read the pilot (Cell 6), then type BUDGET_USD in Cell 5, run Cell 5 again, and run this cell again."
def save(res, name):
    p = store.path(name + ".json"); json.dump(res, open(p, "w")); store.persist(name + ".json"); return p
def persisting(name, inner):
    print("progress file:", store.restore(name + ".jsonl"))
    def cb(*a):
        inner(*a)
        if a[0] % 25 == 0 or a[0] == a[1]: store.persist(name + ".jsonl")
    return cb
def prog(t, n, r):
    if t % 25 == 0 or t == n:
        sp = llm.spent_usd(); c = "summary" if "compaction" in r else "-"
        print(f"step {t}/{n} | prompt {r['prompt_tokens']:,} (cached {r['cache_tokens']:,}) | {r['seconds']:.1f} s | held exact {r['exact']} | reply exact {r['reply_exact']} | last step had a summary: {c} | spent {('$%.2f' % sp) if sp is not None else '-'}")
PRICES = (PRICE_IN, PRICE_OUT, PRICE_CACHED)

### Cell 6: the PILOT. 45 turns of arm S and of arm V on a seed of their own (a few cents to about a dollar). It prints the real size of a heavy turn and of a summary, the first summary the model wrote, and the projected cost of each arm of the whole test at your prices

In [ ]:
assert PAID, "The pilot needs a paid key (about 100 calls)."
ps = go(lambda: C.run_compact(llm, C.SEED_PILOT, C.PILOT_STEPS, "S", store.path("gc_pilot_S.jsonl"), meta=META))
pv = go(lambda: C.run_compact(llm, C.SEED_PILOT, C.PILOT_STEPS, "V", store.path("gc_pilot_V.jsonl"), meta=META))
if ps and pv:
    store.persist("gc_pilot_S.jsonl"); store.persist("gc_pilot_V.jsonl")
    C.print_arm("pilot S", ps["rows"], PRICES); C.print_arm("pilot V", pv["rows"], PRICES)
    for name, res in (("S", ps), ("V", pv)):
        cs = C.compactions(res["rows"]); print(f"\npilot {name}: {len(cs)} summaries; first summary the model wrote ({cs[0]['tokens']} tokens):\n  " + cs[0]["text"][:1500] if cs else f"pilot {name}: NO summary was made: the heavy turns are smaller than expected; tell me before going on")
    print()
    for arm, rows in (("S", ps["rows"]), ("L", ps["rows"]), ("V", pv["rows"])):
        pr = C.project(rows, C.STEPS, C.ARMS[arm]["trigger"], PRICE_IN, PRICE_OUT, PRICE_CACHED)
        print(f"arm {arm} (600 steps) projected:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in pr.items()})
    print("\n'cost' uses your cached price for the cached share the pilot measured; 'cost_nodiscount' uses none. Then type BUDGET_USD in Cell 5 (a little above the three projections added together, or lower: the run stops by itself and what it finished stays valid).")

### Cell 7: ARM S, generic summary at 32,000 tokens. 600 steps. A line every 25 steps. If it stops (budget, busy server, tab closed), run Cells 2 to 5 and this cell again

In [ ]:
need_budget()
nameS = "gc_S_seed%d_%s" % (C.SEED, TAG)
resS = go(lambda: C.run_compact(llm, C.SEED, C.STEPS, "S", store.path(nameS + ".jsonl"), meta=META, max_tokens=C.TOKEN_CAP, progress=persisting(nameS, prog)))
if resS:
    print("stopped because:", resS["stopped"] or "the 600 steps were done")
    C.print_arm("ARM S", resS["rows"], PRICES); store.persist(nameS + ".jsonl")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resS, nameS))

### Cell 8: ARM L, generic summary at 128,000 tokens. 600 steps. A line every 25 steps. If it stops (budget, busy server, tab closed), run Cells 2 to 5 and this cell again

In [ ]:
need_budget()
nameL = "gc_L_seed%d_%s" % (C.SEED, TAG)
resL = go(lambda: C.run_compact(llm, C.SEED, C.STEPS, "L", store.path(nameL + ".jsonl"), meta=META, max_tokens=C.TOKEN_CAP, progress=persisting(nameL, prog)))
if resL:
    print("stopped because:", resL["stopped"] or "the 600 steps were done")
    C.print_arm("ARM L", resL["rows"], PRICES); store.persist(nameL + ".jsonl")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resL, nameL))

### Cell 9: ARM V, the summary writes out the complete current graph, at 32,000 tokens. 600 steps. A line every 25 steps. If it stops (budget, busy server, tab closed), run Cells 2 to 5 and this cell again

In [ ]:
need_budget()
nameV = "gc_V_seed%d_%s" % (C.SEED, TAG)
resV = go(lambda: C.run_compact(llm, C.SEED, C.STEPS, "V", store.path(nameV + ".jsonl"), meta=META, max_tokens=C.TOKEN_CAP, progress=persisting(nameV, prog)))
if resV:
    print("stopped because:", resV["stopped"] or "the 600 steps were done")
    C.print_arm("ARM V", resV["rows"], PRICES); store.persist(nameV + ".jsonl")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resV, nameV))

### Cell 10: the READING. Checks the saved arm A file by its hash, replays every saved reply and every summary (a check that nothing was changed), prints the three arms next to AXI's first 600 steps and the 19 lines with yes / NO, and saves the report. Send me the output and the downloaded files

In [ ]:
apath = "/content/axi/docs/results/axi_gemini_long_session_seed20261111_default.json"
a_ok = hashlib.sha256(open(apath, "rb").read()).hexdigest() == C.A_FILE_SHA256
print("saved arm A of test 22 has the hash written in the code:", a_ok, "(", C.A_FILE_SHA256[:12], ")")
a_rows = json.load(open(apath))["A"]["rows"][:C.STEPS]
arms = dict(S=resS, L=resL, V=resV); ok = {}
for k, res in arms.items():
    try: C.replay(res["rows"], C.SEED, C.ARMS[k]["trigger"], C.ARMS[k]["style"]); ok[k] = True
    except Exception as e: print("replay", k, ":", e); ok[k] = False
try: G.replay(a_rows, C.SEED, "A"); ok["A"] = True
except Exception as e: print("replay A:", e); ok["A"] = False
checks = C.verdict(arms, a_rows, ok, a_ok, PRICES)
print("\nAXI (arm A, its first 600 steps):"); C.print_arm("A", a_rows, PRICES)
for k, res in arms.items(): C.print_arm("arm " + k, res["rows"], PRICES)
C.print_verdict(checks)
for k, res in arms.items():
    cs = C.compactions(res["rows"]); print(f"\narm {k}: {len(cs)} summaries; the first and the last (cut at 800 characters):")
    for c in (cs[:1] + cs[-1:]): print("  ->", c["text"][:800])
print("usage:", llm.usage(), "| spent: $%.2f" % llm.spent_usd())
files.download(save(dict(S=resS, L=resL, V=resV, checks=checks, replays=ok, a_file_ok=a_ok, prices=dict(price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED)), "gemini_compaction_all_%s" % TAG))

### Done
Files downloaded: the arm S, L and V files and `gemini_compaction_all_*.json`. Upload `gemini_compaction_all_default.json` to the chat. Nothing here says anything about any company's own tool: it measures this model on this job.